# Prior-Art Search using RL
## Agentic Search | Multi-Turn | Tool-use  

The objective is to create an Agent to perform prior-art search over patent corpora. mainly The Harvard USPTO Patent Dataset.



In [1]:
from huggingface_hub import notebook_login
import os
import json
import chromadb
from chromadb.api.types import Embeddable, EmbeddingFunction
from chromadb.utils import embedding_functions
import asyncio


In [2]:
# Fields to extract from raw JSON data
RELEVANT_FIELDS = [
    # Identifiers & Linking
    "publication_number",
    "application_number",
    "patent_number",
    # Dates (as epoch ints)
    "date_published",
    "filing_date",
    "patent_issue_date",
    "abandon_date",
    # Status & Classes
    "decision",
    "main_cpc_label",
    "main_ipcr_label",
    # Retrievable Text
    "title",
    "abstract",
    "claims",  ## The legally enforceable boundaries of the invention — the essence of what’s protected.
    # "summary",
]

def get_IP_data():
    """Load and filter IP data from JSON files, skipping files with decode errors."""
    ip_files = []
    for file in os.listdir("Patent_data"):
        file_path = os.path.join("Patent_data", file)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                filtered = {key: value for key, value in data.items() if key in RELEVANT_FIELDS}
                ip_files.append(filtered)
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            print(f"Skipping {file}: {e}")
    return ip_files

patent_data = get_IP_data()

Skipping .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


#### Designing the multi-turn environment

An episode is “one prior-art search task”.


Initial observation is something like:

You are a patent prior-art search assistant.
Here is a new invention description:
<DESCRIPTION>
You have access to tools:

patent_search(query) – searches in a database of patents (title+abstract).

patent_lookup(patent_id) – shows full details of one patent.
Use the tools as needed, then produce a final ranked list of relevant prior-art patents.


The environment passes this as the initial prompt.

After each tool call, the environment appends the tool output to the conversation and gives that back to the model.

3.2. Actions

    Action = “next model output”. You constrain the format so the model can:

    Call tools:
    e.g. CALL_TOOL patent_search: "solar panel with transparent coating for windows"

    Or produce final answer:
    e.g. FINAL_ANSWER: [US1234..., EP5678..., ...] plus explanation.

The environment will:

    Parse the model’s output.

    If it’s CALL_TOOL, execute the underlying Python function and append result.

    If it’s FINAL_ANSWER, end the episode and compute reward.

This is exactly the pattern used in TRL’s experimental “learning tools” examples with calculator/wiki/python environments. 


3.3. Termination

Terminate when:

    The model outputs FINAL_ANSWER, or

    You hit max_steps (e.g. 5–8 tool calls) and then force the model to give a final answer, or mark as failure.

In [3]:
CHROMA_DB_DIR = ".chroma_db"
_chroma_semaphore: asyncio.Semaphore | None = None

def get_chroma_semaphore() -> asyncio.Semaphore:
    global _chroma_semaphore
    if _chroma_semaphore is None:
        _chroma_semaphore = asyncio.Semaphore(20)
    return _chroma_semaphore


def init_chroma_collection(force_recreate: bool = False) :
    """Connect to an existing ChromaDB collection for patent data, or create if not exists."""
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
    # Try to get the collection if it exists
    try:
        if force_recreate:
            chroma_client.delete_collection(name="patent_collection")
        collection = chroma_client.get_collection(
            name="patent_collection",
            embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name="sentence-transformers/all-mpnet-base-v2"
            )
        )
        # If collection exists, return it
        return collection
    except Exception:
        # If not found, create it and add data
        collection = chroma_client.get_or_create_collection(
            name="patent_collection",
            embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name="sentence-transformers/all-mpnet-base-v2"
            ),
            configuration={
                "hnsw": {
                    "space": "cosine",
                    "ef_construction": 200,
                    "ef_search": 150
                }
            }
        )
        collection.add(
            documents=[patent["abstract"] for patent in patent_data],  # Using abstract as the main text for embeddings
            ids=[patent["publication_number"] for patent in patent_data],
            metadatas=[{k: v for k, v in patent.items() if k != 'abstract'} for patent in patent_data],
        )
        return collection

collection = init_chroma_collection(force_recreate=False)


In [4]:
collection.get("US20180137422A1-20180517")

{'ids': ['US20180137422A1-20180517'],
 'embeddings': None,
 'documents': ['Methods of training Boltzmann machines include rejection sampling to approximate a Gibbs distribution associated with layers of the Boltzmann machine. Accepted sample values obtained using a set of training vectors and a set of model values associate with a model distribution are processed to obtain gradients of an objective function so that the Boltzmann machine specification can be updated. In other examples, a Gibbs distribution is estimated or a quantum circuit is specified so at to produce eigenphases of a unitary.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'date_published': '20180517',
   'application_number': '15579190',
   'patent_number': 'None',
   'patent_issue_date': '',
   'title': 'FAST LOW-MEMORY METHODS FOR BAYESIAN INFERENCE, GIBBS SAMPLING AND DEEP LEARNING',
   'decision': 'PENDING',
   'main_ipcr_label': 'G06N502',
   'filing_date': '20171201',
 

### Defining The Tools

In [5]:
from pydantic import BaseModel, Field
from dataclasses import asdict, dataclass
from typing import List, Optional


## search tool
async def search_patents(query: str, n_results: int = 10) -> list[dict]:
    """Search for top 10 relevant patents using title embedding similarity."""
    async with get_chroma_semaphore():
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
    
    if not results:
        raise ValueError(f"No results found for query: {query}")
    if not results["metadatas"]:
        raise ValueError(f"No results metadata found for query: {query}")
    
    output = []
    for i in range(len(results["ids"][0])):
        patent_title = results["metadatas"][0][i]["title"]
        publication_number = results["ids"][0][i]
        similarity_score = results["distances"][0][i]
        output.append({"patent_title": patent_title, 
                       "publication_number": publication_number,
                       "similarity_score": similarity_score})
    
    return output

# Patent lookup tool
async def lookup_patent(publication_number: str) -> dict:
    """Lookup patent details by publication number."""
    sem = get_chroma_semaphore()
    async with sem:
        results = await asyncio.to_thread(
            collection.get,
            ids=[publication_number],
        )

    if not results or not results.get("metadatas"):
        raise ValueError(f"No patent found with publication number: {publication_number}")

    patent_content = results["documents"][0]
    patent_metadata = results["metadatas"][0]
    return {**patent_metadata, "abstract": patent_content}


def return_final_answer(
        answer: str, 
        reference_message_ids: list[str]) -> FinalAnswer:
        """Return the final answer and the message IDs of the emails that were used to generate the answer."""
        return FinalAnswer(answer=answer, source_ids=reference_message_ids)

In [6]:
await search_patents("battery management", n_results=3)

[{'patent_title': 'QUICK-EXCHANGE BATTERY ASSEMBLY, AND MOTOR VEHICLE, IN PARTICULAR MOTOR SCOOTER',
  'publication_number': 'US20180151860A1-20180531',
  'similarity_score': 0.5740572214126587},
 {'patent_title': 'METHOD FOR CALCULATING A SETPOINT FOR MANAGING THE FUEL AND ELECTRICITY CONSUMPTION OF A HYBRID MOTOR VEHICLE',
  'publication_number': 'US20180281620A1-20181004',
  'similarity_score': 0.6117575168609619},
 {'patent_title': 'POSITIVE ELECTRODE ACTIVE MATERIAL FOR RECHARGABLE LITHIUM BATTERY, METHOD FOR MANUFACTURING SAME, AND RECHARGABLE LITHIUM BATTERY INCLUDING SAME',
  'publication_number': 'US20180254473A1-20180906',
  'similarity_score': 0.6518891453742981}]

## Reward Design for prior-art Search

### Backward-citation

We need labels for what good prior art looks like.
since we dont have 'citations' in the documents


the ground truth will be basically patents abstract snippets and their backward citations as ground truth.
For a given patent P, the cited patents = “true” prior art for P.


In [7]:
import dspy
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
base_url = os.getenv("GEMINI_API_BASE", "https://generativelanguage.googleapis.com/v1beta")
api_key = os.getenv("GEMINI_API_KEY")
lm = dspy.LM('gemini/gemini-2.0-flash', api_key=api_key)
dspy.configure(lm=lm)

class ExtractInfo(dspy.Signature):
    """you are gonna be given an abtract of a patent and you need to generate 3 queries that
    can be used to search for prior art patents related to the given patent abstract. 
    The queries should be concise and relevant to the key aspects of the patent abstract.
    only return the queries as a json array of strings with no other text."""

    patent_abstract: str = dspy.InputField()
    number_of_queries: int = dspy.InputField(default=3, desc="Number of search queries to generate.")
    queries: list[str] = dspy.OutputField(desc="List of search queries for prior art patents.")

module = dspy.Predict(ExtractInfo)

def get_queries_from_abstract(patent_abstract: str, number_of_queries=3) -> list[str]:
    """Generate search queries for prior art patents based on the given patent abstract."""
    try:
        return module(patent_abstract=patent_abstract, 
                      number_of_queries=number_of_queries).queries
    except Exception as e:
        raise print(f"Error generating queries: {e}")
        
    
def create_search_queries(patent_data: list[str], number_of_queries=3)-> pd.DataFrame:
    """Generate search queries for a list of patent abstracts."""
    all_queries = []
    for patent in patent_data:
        queries = get_queries_from_abstract(patent['abstract'], number_of_queries)
        for query in queries:
            all_queries.append({'publication_number': patent['publication_number'], 'query': query, 'abstract': patent['abstract']})
    out = pd.DataFrame(all_queries)
    out.to_csv("Evals/patent_search_queries.csv", index=False)
    return out
# df = create_search_queries(patent_data, number_of_queries=3)

In [14]:
patent_search_queries = pd.read_csv("Evals/patent_search_queries.csv")
patent_search_queries


,publication_number,query,abstract
0,US20180271722A1-20180927,disposable absorbent article vital signs monit...,The present invention protects a system for mo...
1,US20180271722A1-20180927,sensor device bi-dimensional code vital signs,The present invention protects a system for mo...
2,US20180271722A1-20180927,wearable sensor absorbent article health monit...,The present invention protects a system for mo...
3,US20180223520A1-20180809,prefabricated modular building hyperstatic frame,The invention relates to a prefabricated modul...
4,US20180223520A1-20180809,self-supporting structural frame connection sy...,The invention relates to a prefabricated modul...
...,...,...,...
2071,US20180115259A1-20180426,Actuator body rotatably guided on base body se...,"Servomotor (1) comprising: —a base body (10), ..."
2072,US20180115259A1-20180426,Magnet-compensation device for servomotor rese...,"Servomotor (1) comprising: —a base body (10), ..."
2073,US20180170476A1-20180621,two-wheel dynamic balance vehicle,A two-wheel dynamic balance vehicle comprises ...
2074,US20180170476A1-20180621,self-balancing scooter steering mechanism,A two-wheel dynamic balance vehicle comprises ...


In [ ]:
from openai import OpenAI, AsyncOpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
### model needs to be served via vLLM
client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="whatever"  
)

try:
    response = await client.chat.completions.create(
        model="Qwen/Qwen3-0.6B",  # Use the correct model name from vLLM
        messages=[{"role": "user", "content": "Hello!"}],
    )
    print("Success!")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print(f"Message: {str(e)}")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `mistral` does not exist.', 'type': 'NotFoundError', 'param': None, 'code': 404}}

In [ ]:
MAX_TURNS = 6
# Patent data models
class Patent(BaseModel):
    publication_number: str
    title: str  
    abstact: Optional[str] = None
    main_ipcr_label: Optional[str] = None
    main_cpc_label: List[str] = []  
    decision: List[str] = []  
    patent_issue_date: List[str] = []  


@dataclass
class SearchResult:
    message_id: str
    snippet: str

class FinalAnswer(BaseModel):   
    answer: str
    patent_ids: list[str]


async def rollout(model, SearchScenario):
    scenario = SearchScenario
    system_prompt ="""
        You are a prior-art search agent. You are given a new invention description and a list of tools you can use to perform prior-art search. 
        Use the tools to search for relevant prior patent, and find the patent . 
        You may take up to {MAX_TURNS} turns to find the answer, so if your first search doesn't find the answer, you can try with different queries.

        Tools:
        1. search_patents(query: str, n_results: int = 10) -> list[dict]: 
           Search for top n_results relevant patents using title embedding similarity.
        2. lookup_patent(publication_number: str) -> dict:
            Lookup patent details by publication number.
        3. return_final_answer(answer: str, reference_message_ids: list[str]) -> FinalAnswer:
            Return the final answer and the list of patent publication numbers that were used to generate the answer

        User's new invention description: {scenario.invention_description}
        """

    tools = [search_patents, lookup_patent, return_final_answer]
    tools_by_name = {t.__name__: t for t in tools}

    client = AsyncOpenAI(
        base_url=model.inference_base_url,
        api_key=model.inference_api_key,
    )

    for _ in range(MAX_TURNS):
        response = await client.chat.completions.create(
            model=model.get_inference_name(),
            temperature=1,
            messages=traj.messages(),
            tools=traj.tools,
        )

        response_message = response.choices[0].message
        traj.messages_and_choices.append(response.choices[0])

        if not response_message.tool_calls:
            return traj

        try:
            for tool_call in response_message.tool_calls:
                tool_name: str = tool_call.function.name
                if tool_name in tools_by_name:
                    tool_args = json.loads(tool_call.function.arguments)
                    tool_to_call = tools_by_name[tool_name]
                    result = tool_to_call(**tool_args)
                    traj.messages_and_choices.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": tool_name,
                            "content": str(result),
                        }
                    )

                    if tool_name == "return_final_answer":
                        traj.final_answer = result
                        # Score the trajectory
                        if traj.final_answer:
                            correctness_judge_response = await judge_correctness(
                                scenario, traj.final_answer.answer
                            )
                            traj.metrics["correct"] = float(
                                correctness_judge_response.accept
                            )
                        return traj
        except Exception as e:
            print(f"Error executing tool call: {e}")
            return traj

    return traj

### RULER:  LLM-as-a-judge
As in RULER from openpipe


RULER (Relative Universal LLM-Elicited Rewards) is a general-purpose reward function that uses an LLM-as-judge to rank multiple agent trajectories. 

It requires no labeled data, expert feedback, or hand-crafted reward functions, yet reliably improves agent performance.